# Análisis Exploratorio de Datos: EcoBici CDMX

El sistema **EcoBici** es una de las fuentes de datos abiertos más ricas de la Ciudad de México. A diferencia de otros datasets estáticos, este nos permite analizar la **dinámica urbana** a través del tiempo y el espacio.

## Objetivos del Notebook
1. **Procesamiento de Fechas**: Aprender a manejar formatos `datetime`.
2. **Limpieza de Outliers**: Identificar viajes inválidos o técnicos.
3. **Patrones Temporales**: ¿A qué hora se usa más el sistema? ¿Cómo cambia el fin de semana?
4. **Análisis de Origen-Destino**: Identificar las estaciones con mayor demanda.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Carga de Datos

Utilizaremos un archivo mensual de los datos abiertos de la CDMX. 
> **Nota**: Estos archivos suelen ser grandes (>100MB). Para este ejercicio, puedes descargar uno directamente de: [Datos Abiertos CDMX](https://ecobici.cdmx.gob.mx/datos-abiertos/?gad_source=1&gad_campaignid=23606083357&gbraid=0AAAAA_xfLRMWRYYEEcXy4H5OkTwa61_bX&gclid=CjwKCAiAzZ_NBhAEEiwAMtqKy0W0-5T-8bSJuHWIbLFwQc_-JzKeAB3_ITlPlGvu2Hii05-1Ymb5uRoC0MEQAvD_BwE)

In [ ]:
# URL de ejemplo (Enero 2026)
#df = pd.read_csv(
#    "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/02/2026-01.csv",
#    low_memory=True
#)

df = pd.read_csv(
    "../data/2026-01.csv",
    low_memory=True
)

In [ ]:
df.info(memory_usage='deep')

In [ ]:
df

In [ ]:
df_clean = (
    df
    .assign(
        inicio = lambda x: pd.to_datetime(x.Fecha_Retiro + " " + x.Hora_Retiro, dayfirst=True),
        fin = lambda x: pd.to_datetime(x.Fecha_Arribo + " " + x.Hora_Arribo, dayfirst=True)
    )
    .assign(
        duracion_viaje = lambda x: x.fin - x.inicio,
        dia_semana = lambda x: x.inicio.dt.day_name()
    )
)

In [ ]:
df_clean

¿Se les ocurren más variables? ¿Cuáles? ¿Para qué?

# ¿Quienes son los usuarios de ecobici?

## ¿Cuál es el género qué más utiliza la ecobici?

In [ ]:
sns.barplot(
    data=, # ¿Qué conjunto de datos necesitamos aquí?
    x=, # ¿Cuál será el eje x?
    y=, # ¿Cuál será el eje y?
)

## ¿Cuáles son las edades de los usuarios?

In [ ]:
sns.displot(
    data=, # ¿Qué conjunto de datos necesitamos aquí? ¿Necesitamos agregar?
    x=,
)

# ¿Cómo usan la ecobici?

## ¿Cuánto dura cada viaje?

In [ ]:
sns.displot(
    data=df_clean.assign(duracion_minutos=lambda x: x.duracion_viaje.dt.total_seconds() / 60),
    x='duracion_minutos',
)

¿Qué está pasando? ¿Cómo podemos resolverlo?

## ¿Cuántas veces usa el usuario promedio la ecobici?

In [ ]:
# ¿Necesitamos agrupar para responder esta pregunta?
# Si es así, ¿por qué columna agrupamos y qué calculamos?

viajes_por_usuario = (
    df_clean
    .groupby(_____)  # ¿Qué identifica a un usuario?
    .agg(
        num_viajes = (_____,  _____)  # ¿Qué columna contamos y con qué función?
    )
)

viajes_por_usuario.describe()

# ¿Cuándo usan la ecobici?

Hasta ahora hemos analizado **quiénes** son los usuarios. Ahora queremos entender **cuándo** usan el sistema.

> **Reflexión previa**: Para cada pregunta que sigue, piensen antes de escribir código:
> - ¿Necesito agrupar los datos o puedo graficar directamente?
> - Si agrupo, ¿por qué columna(s)?
> - ¿Qué calculo dentro de cada grupo?

## ¿A qué hora del día se usa más la ecobici?

In [ ]:
# Primero extraemos la hora de inicio (ya tenemos la columna `inicio`)
df_hora = df_clean.assign(
    hora = _____  # Hint: dt.hour
)

# ¿Necesitamos agrupar? ¿O podemos pasar df_hora directamente a seaborn?
viajes_por_hora = (
    df_hora
    .groupby(_____)
    .agg(num_viajes = (_____, _____))
    .reset_index()
)

sns.barplot(
    data=viajes_por_hora,
    x=_____,
    y=_____,
)

## ¿El patrón de uso por hora es igual entre semana y fin de semana?

Ahora la pregunta es más compleja: queremos ver la hora **y** el tipo de día al mismo tiempo.

> **Reflexión**: ¿Cómo cambia el `groupby` respecto al ejercicio anterior? ¿Por cuántas columnas agrupamos ahora?

In [ ]:
FINES_DE_SEMANA = ['Saturday', 'Sunday']

df_tipo_dia = df_clean.assign(
    hora = _____,
    tipo_dia = lambda x: x.dia_semana.apply(
        lambda d: _____ if d in FINES_DE_SEMANA else _____  # 'Fin de semana' o 'Entre semana'
    )
)

viajes_hora_tipo = (
    df_tipo_dia
    .groupby([_____, _____])  # ¿Por cuáles dos columnas agrupamos?
    .agg(num_viajes = (_____, _____))
    .reset_index()
)

# ¿Cómo diferenciamos visualmente entre semana y fin de semana en la gráfica?
sns.lineplot(
    data=viajes_hora_tipo,
    x=_____,
    y=_____,
    hue=_____,  # ¿Qué columna usamos para diferenciar los grupos?
)

# ¿Cómo difiere el uso entre grupos de usuarios?

Ahora combinamos las dimensiones de **quién** y **cómo** usa la ecobici.

> **Reflexión**: ¿Qué pasa cuando agrupamos por una variable categórica y queremos resumir una variable numérica? ¿Qué función de agregación tiene sentido usar?

## ¿La duración promedio del viaje difiere según el género del usuario?

In [ ]:
df_duracion = df_clean.assign(
    duracion_minutos = lambda x: x.duracion_viaje.dt.total_seconds() / 60
)

# Antes de graficar: ¿Qué valores de Genero_Usuario existen?
# ¿Necesitamos filtrar alguno?
print(df_duracion['Genero_Usuario'].value_counts())

duracion_por_genero = (
    df_duracion
    # Opcional: filtrar géneros con pocos registros o valores inesperados
    # .query(_____)
    .groupby(_____)
    .agg(
        duracion_promedio = (_____, _____),  # ¿Media, mediana? ¿Por qué?
        duracion_mediana  = (_____, _____),
    )
    .reset_index()
)

duracion_por_genero

## ¿La hora de uso varía según el género?

> **Reflexión**: Esta pregunta requiere agrupar por **dos variables** simultáneamente. ¿Cómo afecta esto la forma del DataFrame resultante y cómo lo visualizamos?

In [ ]:
viajes_hora_genero = (
    df_clean
    .assign(hora = _____)
    .query("Genero_Usuario in ['M', 'F']")  # Filtramos valores válidos
    .groupby([_____, _____])
    .agg(num_viajes = (_____, _____))
    .reset_index()
)

sns.lineplot(
    data=viajes_hora_genero,
    x=_____,
    y=_____,
    hue=_____,
)

# ¿Dónde usan la ecobici?

## ¿Cuáles son las estaciones con mayor demanda?

> **Reflexión**: Tenemos dos columnas de estación: `Ciclo_Estacion_Retiro` (origen) y `Ciclo_EstacionArribo` (destino). ¿Qué pregunta responde cada una? ¿Qué significa que una estación tenga muchos retiros pero pocos arribos?

In [ ]:
TOP_N = 15

# Estaciones con más viajes iniciados (retiros)
top_retiros = (
    df_clean
    .groupby(_____)
    .agg(num_retiros = (_____, _____))
    .reset_index()
    .sort_values(_____, ascending=_____)  # ¿Ascendente o descendente?
    .head(TOP_N)
)

# Estaciones con más viajes terminados (arribos)
top_arribos = (
    df_clean
    .groupby(_____)
    .agg(num_arribos = (_____, _____))
    .reset_index()
    .sort_values(_____, ascending=_____)
    .head(TOP_N)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=top_retiros, y=_____, x=_____, ax=axes[0])
axes[0].set_title("Top estaciones de retiro")

sns.barplot(data=top_arribos, y=_____, x=_____, ax=axes[1])
axes[1].set_title("Top estaciones de arribo")

plt.tight_layout()

## ¿Cuáles son los pares origen-destino más frecuentes?

> **Reflexión**: Ahora agrupamos por **dos columnas de categorías** al mismo tiempo. El resultado ya no es una lista de estaciones, sino una tabla de pares. ¿Cómo cambia la interpretación?

In [ ]:
pares_od = (
    df_clean
    .groupby([_____, _____])  # origen y destino
    .agg(num_viajes = (_____, _____))
    .reset_index()
    .sort_values(_____, ascending=False)
    .head(20)
)

pares_od

¿Hay viajes donde el origen y el destino son la **misma estación**? ¿Qué podrían significar? ¿Deberíamos incluirlos en nuestro análisis?

# ¿Dónde están las estaciones de EcoBici?

Contamos con un segundo dataset que tiene la **ubicación geográfica** de cada cicloestación. Para poder usarlo junto con los viajes, necesitamos **unir** ambas tablas.

> **Reflexión previa**: 
> - ¿Qué columna del dataset de viajes se relaciona con el dataset de estaciones?
> - ¿Necesitamos el mismo `groupby` de antes antes de hacer el join, o podemos unir directamente?

In [ ]:
import plotly.express as px

estaciones = pd.read_csv("https://datos.cdmx.gob.mx/dataset/a1d7c132-fb1b-4e8c-bb74-4bb618563eb2/resource/5fbacfcc-f677-406c-9356-6ced541240fe/download/cicloestaciones_ecobici.csv")
estaciones.head()

## Mapa base: ¿dónde está cada estación?

Antes de cruzar con los viajes, grafiquemos simplemente la ubicación de todas las estaciones.

In [ ]:
px.scatter_map(
    data_frame=estaciones,
    lat=_____,       # columna con latitud
    lon=_____,       # columna con longitud
    hover_name=_____, # ¿qué queremos ver al pasar el mouse?
    zoom=11,
    height=500,
)

## ¿Cuáles estaciones tienen más retiros?

Ahora queremos que el **tamaño** y **color** de cada punto refleje cuántos viajes se iniciaron en esa estación.

Para lograrlo necesitamos:
1. Agregar los viajes por estación de retiro (ya lo hicimos antes)
2. **Unir** ese resultado con el dataset de estaciones (para obtener lat/lon)

> **Reflexión**: ¿Qué tipo de join necesitamos? ¿`inner`, `left`, `right`? ¿Qué pasa si una estación tiene viajes pero no aparece en el dataset de ubicaciones, o viceversa?

In [ ]:
retiros_por_estacion = (
    df_clean
    .groupby(_____)                             # ¿Por qué columna agrupamos?
    .agg(num_retiros=(_____, _____))
    .reset_index()
    .rename(columns={_____: 'num_cicloe'})      # Igualamos el nombre de la llave
)

# Unimos con las coordenadas
mapa_retiros = retiros_por_estacion.merge(
    estaciones[['num_cicloe', 'latitud', 'longitud', 'calle_prin', 'alcaldia']],
    on=_____,   # ¿Cuál es la llave?
    how=_____,  # ¿Qué tipo de join usamos?
)

mapa_retiros.head()

In [ ]:
px.scatter_map(
    data_frame=mapa_retiros,
    lat=_____,
    lon=_____,
    size=_____,           # ¿Qué columna determina el tamaño del punto?
    color=_____,          # ¿Y el color?
    hover_name=_____,
    hover_data={'alcaldia': True, 'num_retiros': True, 'latitud': False, 'longitud': False},
    color_continuous_scale='Reds',
    size_max=30,
    zoom=11,
    height=600,
    title='Estaciones de EcoBici: demanda de retiros',
)

## ¿Hay estaciones que "expulsan" más bicicletas de las que reciben?

Una estación con muchos **retiros** y pocos **arribos** pierde bicicletas durante el día (piensen en las estaciones cerca del metro por las mañanas). El **flujo neto** = arribos − retiros nos dice qué estaciones acumulan o pierden bicicletas.

> **Reflexión**: ¿Cómo construimos esta tabla? ¿Cuántos `groupby` necesitamos y cómo los combinamos?

In [ ]:
retiros = (
    df_clean
    .groupby('Ciclo_Estacion_Retiro')
    .agg(num_retiros=('Bici', 'count'))
    .reset_index()
    .rename(columns={'Ciclo_Estacion_Retiro': 'num_cicloe'})
)

arribos = (
    df_clean
    .groupby(_____)
    .agg(num_arribos=(_____, _____))
    .reset_index()
    .rename(columns={_____: 'num_cicloe'})
)

# Unimos retiros y arribos en una sola tabla por estación
flujo = (
    retiros
    .merge(arribos, on=_____, how=_____)
    .assign(flujo_neto = _____) # arribos - retiros
    .merge(
        estaciones[['num_cicloe', 'latitud', 'longitud', 'calle_prin', 'alcaldia']],
        on='num_cicloe',
        how='inner',
    )
)

flujo.sort_values('flujo_neto').head(10)  # Estaciones que más pierden bicicletas

In [ ]:
px.scatter_map(
    data_frame=flujo,
    lat='latitud',
    lon='longitud',
    color=_____,                              # flujo_neto: negativo = pierde, positivo = acumula
    hover_name=_____,
    hover_data={'alcaldia': True, 'flujo_neto': True, 'latitud': False, 'longitud': False},
    color_continuous_scale=_____,             # Hint: 'RdBu' diferencia positivos y negativos
    color_continuous_midpoint=_____,          # ¿Cuál debería ser el punto medio de la escala?
    size=flujo['flujo_neto'].abs(),           # El tamaño muestra la magnitud del desequilibrio
    size_max=25,
    zoom=11,
    height=600,
    title='Flujo neto de bicicletas por estación (arribos − retiros)',
)

## ¿Podemos ver los flujos más frecuentes como líneas en el mapa?

Para dibujar una línea entre dos puntos en `plotly`, necesitamos que cada línea esté representada por **dos filas consecutivas** en el DataFrame (origen y destino), identificadas por un mismo grupo.

> **Reflexión**: Hasta ahora cada fila representaba una estación. Ahora necesitamos que cada par O-D ocupe **dos filas**. ¿Cómo transformamos los datos para llegar a esa estructura?

In [ ]:
TOP_N = 30

# Paso 1: obtener los pares O-D más frecuentes con un ID único por par
pares_top = (
    df_clean
    .groupby([_____, _____])
    .agg(num_viajes=(_____, _____))
    .reset_index()
    .sort_values('num_viajes', ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
    .assign(par_id=lambda x: x.index)  # ID único para cada par
)

coords = estaciones[['num_cicloe', 'latitud', 'longitud', 'calle_prin']]

# Paso 2: unir coordenadas del ORIGEN
origen = (
    pares_top
    .merge(
        coords.rename(columns={'num_cicloe': _____, 'latitud': 'lat', 'longitud': 'lon', 'calle_prin': 'nombre'}),
        on=_____,
        how='left',
    )
)

# Paso 3: unir coordenadas del DESTINO
destino = (
    pares_top
    .merge(
        coords.rename(columns={'num_cicloe': _____, 'latitud': 'lat', 'longitud': 'lon', 'calle_prin': 'nombre'}),
        on=_____,
        how='left',
    )
)

# Paso 4: concatenar ambos — cada par queda en dos filas consecutivas
lineas = (
    pd.concat([origen, destino])
    .sort_values(_____)  # ¿Por qué columna ordenamos para que plotly conecte los puntos correctamente?
)

lineas.head(6)

In [ ]:
px.line_map(
    data_frame=lineas,
    lat=_____,
    lon=_____,
    line_group=_____,      # ¿Qué columna agrupa los dos puntos de cada línea?
    color=_____,           # ¿Qué columna usamos para indicar la frecuencia del par?
    hover_name=_____,
    zoom=11,
    height=600,
    title=f'Top {TOP_N} pares origen-destino más frecuentes',
)

## Calcular distancias entre estaciones de ecobici

Si ya tienes scikit-learn instalado (muy probable en cualquier entorno de data science), puedes usarla sin instalar nada extra. La única diferencia: **requiere coordenadas en radianes**.

> **Reflexión**: ¿Por qué crees que sklearn pide radianes en lugar de grados? ¿Qué tiene que hacer internamente con las coordenadas?

In [ ]:
from sklearn.metrics.pairwise import haversine_distances

RADIO_TIERRA_KM = 6_371

# sklearn requiere radianes y formato (lat, lon) como filas de una matriz
coords_rad = np.radians(estaciones[['latitud', 'longitud']].values)

# El resultado está en radianes → multiplicamos por el radio de la Tierra
matriz_sklearn = haversine_distances(coords_rad) * RADIO_TIERRA_KM

print(f"Forma de la matriz : {matriz_sklearn.shape}")
print(f"\nPrimeras 5×5 distancias (km):")
print(np.round(matriz_sklearn[:5, :5], 2))

## ¿Cómo usamos estos resultados en la base original?
¿Cómo se imaginan el uso?